# 5. Interactive Visual Analytics — Folium & Plotly Dash

Builds an interactive Folium map of launch sites and outcomes, and a Plotly Dash
dashboard with a site dropdown and payload-mass range slider.

> **Note:** Folium and Dash render interactively in a browser/notebook kernel and
> don't have a static "output" the way a printed value does — run this notebook
> live to interact with the map and dashboard. Static preview images generated
> from this same data are included in `../assets/` and in the final presentation.

## 5.1 Folium map of launch sites

In [ ]:
import folium
from folium.plugins import MarkerCluster
import pandas as pd

df = pd.read_csv('../data/spacex_launch_data.csv')

site_coords = {
    'CCAFS SLC 40': (28.5618571, -80.577366),
    'KSC LC 39A':   (28.6080585, -80.6039558),
    'VAFB SLC 4E':  (34.632093, -120.6108290),
}

nasa_map = folium.Map(location=[29, -95], zoom_start=4)

for site, (lat, lon) in site_coords.items():
    rate = df.loc[df['LaunchSite'] == site, 'Class'].mean()
    folium.Circle(
        [lat, lon], radius=1000, color='#3D8BFF', fill=True,
        popup=f"{site}: {rate:.0%} success rate"
    ).add_to(nasa_map)
    folium.map.Marker(
        [lat, lon],
        icon=folium.DivIcon(html=f'<div style="font-size:12px;color:#0B1F3A;"><b>{site}</b></div>')
    ).add_to(nasa_map)

# Individual launch outcome markers, clustered per site
marker_cluster = MarkerCluster().add_to(nasa_map)
for _, row in df.iterrows():
    lat, lon = row['Latitude'], row['Longitude']
    color = 'green' if row['Class'] == 1 else 'red'
    folium.Marker(
        [lat, lon],
        icon=folium.Icon(color=color, icon='rocket', prefix='fa')
    ).add_to(marker_cluster)

nasa_map.save('../assets/launch_sites_map.html')
nasa_map

## 5.2 Plotly Dash dashboard

In [ ]:
import dash
from dash import dcc, html, Input, Output
import plotly.express as px
import pandas as pd

df = pd.read_csv('../data/spacex_launch_data.csv')

app = dash.Dash(__name__)

app.layout = html.Div([
    html.H1('SpaceX Launch Records Dashboard'),
    dcc.Dropdown(
        id='site-dropdown',
        options=[{'label': 'All Sites', 'value': 'ALL'}] +
                [{'label': s, 'value': s} for s in df['LaunchSite'].unique()],
        value='ALL',
    ),
    dcc.Graph(id='success-pie-chart'),
    dcc.RangeSlider(
        id='payload-slider', min=0, max=16000, step=1000,
        value=[0, 16000],
        marks={i: str(i) for i in range(0, 16001, 4000)},
    ),
    dcc.Graph(id='success-payload-scatter-chart'),
])

@app.callback(
    Output('success-pie-chart', 'figure'),
    Input('site-dropdown', 'value'),
)
def update_pie(selected_site):
    if selected_site == 'ALL':
        return px.pie(df, names='LaunchSite', values='Class', title='Total Successful Landings by Site')
    filtered = df[df['LaunchSite'] == selected_site]
    return px.pie(filtered, names='Class', title=f'Success vs. Failure — {selected_site}')

@app.callback(
    Output('success-payload-scatter-chart', 'figure'),
    Input('site-dropdown', 'value'),
    Input('payload-slider', 'value'),
)
def update_scatter(selected_site, payload_range):
    low, high = payload_range
    filtered = df[(df['PayloadMass'] >= low) & (df['PayloadMass'] <= high)]
    if selected_site != 'ALL':
        filtered = filtered[filtered['LaunchSite'] == selected_site]
    return px.scatter(filtered, x='PayloadMass', y='Class', color='BoosterVersion',
                       title='Payload Mass vs. Landing Outcome')

if __name__ == '__main__':
    app.run_server(debug=True)

**Dashboard behavior:**
- Selecting a launch site in the dropdown updates the pie chart to that site's success/failure split; "All Sites" shows the share of total successful landings contributed by each site (KSC LC-39A contributes the most, consistent with its ~77% success rate).
- Moving the payload slider filters both charts live, showing that landings cluster at the lower end of the payload range and drop off above ~10,000 kg.